## V2 Model Dataset

Makes following changes to 01_regression_data:
1. Decomposed Price (Base Energy + Per-Component Adders)
2. Replace cap_threshold with 1/reserve_margin
3. Include outage in the spare_capacity matric calculation

Scope: 2021-01 to 2025-12-04 (pre-RTC+B), excluding Winter Storm Uri

In [6]:
import pandas as pd
import numpy as np

MODEL_DATA = '../../01_data/4_model/raw_model_dataset.csv'
ADDER_PATH = '../../01_data/2_cleaned/price_adders/price_adders_hourly_20200101_20251204.csv'
OUT_PATH   = '../../01_data/4_model/v2_model_dataset.csv'

### Load cleaned hourly adder data

In [7]:
hourly_adder = pd.read_csv(ADDER_PATH)
hourly_adder['datetime'] = pd.to_datetime(hourly_adder['datetime'])

# Filter to 2021+ (model dataset starts 2021)
hourly_adder = hourly_adder[hourly_adder['datetime'] >= '2021-01-01']

hourly_adder['total_adder'] = (
    hourly_adder['RTRSVPOR'] +
    hourly_adder['RTRSVPOFF'] +
    hourly_adder['RTRDP']
)
hourly_adder

,datetime,RTRSVPOR,RTRSVPOFF,RTRDP,total_adder
8782,2021-01-01 00:00:00,0.0,0.0,0.00,0.00
8783,2021-01-01 01:00:00,0.0,0.0,0.00,0.00
8784,2021-01-01 02:00:00,0.0,0.0,0.00,0.00
8785,2021-01-01 03:00:00,0.0,0.0,0.00,0.00
8786,2021-01-01 04:00:00,0.0,0.0,0.00,0.00
...,...,...,...,...,...
51924,2025-12-03 19:00:00,0.0,0.0,0.01,0.01
51925,2025-12-03 20:00:00,0.0,0.0,0.00,0.00
51926,2025-12-03 21:00:00,0.0,0.0,0.00,0.00
51927,2025-12-03 22:00:00,0.0,0.0,0.00,0.00


### Merge into full dataset

In [8]:
df_model = pd.read_csv(MODEL_DATA)
df_model['datetime'] = pd.to_datetime(df_model['datetime'])

df = pd.merge(df_model, hourly_adder, on='datetime', how='inner')
print(f'Merged: {len(df)} rows (model had {len(df_model)})')

# Exclude Winter Storm Uri (same as v1)
uri_mask = (
    (df['datetime'] >= '2021-02-13') &
    (df['datetime'] <= '2021-02-25')
)
print(f'Uri rows removed: {uri_mask.sum()}')
df = df[~uri_mask]
df

Merged: 42883 rows (model had 43531)
Uri rows removed: 25


,datetime,year,time_of_day,peak_hour,season,rtm_price,total_load,perc_load,spare_capacity,ng_price_monthly,degree_days,heating_dd,cap_threshold,RTRSVPOR,RTRSVPOFF,RTRDP,total_adder
0,2021-01-01 01:00:00,2021,1,False,winter,18.173500,43713.80,0.594948,0.447794,2.712632,29.698555,False,False,0.0,0.0,0.00,0.00
1,2021-01-01 02:00:00,2021,2,False,winter,20.960833,43314.37,0.589511,0.437193,2.712632,30.076485,False,False,0.0,0.0,0.00,0.00
2,2021-01-01 03:00:00,2021,3,False,winter,20.779667,43048.35,0.585891,0.434211,2.712632,30.348450,False,False,0.0,0.0,0.00,0.00
3,2021-01-01 04:00:00,2021,4,False,winter,16.416667,43073.65,0.586235,0.449278,2.712632,31.037514,False,False,0.0,0.0,0.00,0.00
4,2021-01-01 05:00:00,2021,5,False,winter,18.075333,43448.89,0.591342,0.470151,2.712632,32.820100,False,False,0.0,0.0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42878,2025-12-03 19:00:00,2025,19,True,winter,37.520333,54497.98,0.649729,0.591470,4.263810,9.542927,False,False,0.0,0.0,0.01,0.01
42879,2025-12-03 20:00:00,2025,20,True,winter,40.374667,54169.25,0.645810,0.552325,4.263810,10.943300,False,False,0.0,0.0,0.00,0.00
42880,2025-12-03 21:00:00,2025,21,True,winter,38.039500,53803.77,0.641453,0.529319,4.263810,12.875572,False,False,0.0,0.0,0.00,0.00
42881,2025-12-03 22:00:00,2025,22,True,winter,34.266667,52867.06,0.630285,0.498086,4.263810,13.614403,False,False,0.0,0.0,0.00,0.00


### Derive base energy and activation flags

In [4]:
df['base_energy'] = df['rtm_price'] - df['total_adder']

# Per-component activation flags
df['rtrsvpor_active'] = (df['RTRSVPOR'] > 0).astype(int)
df['rtrsvpoff_active'] = (df['RTRSVPOFF'] > 0).astype(int)
df['rtrdp_active'] = (df['RTRDP'] > 0).astype(int)

print('=== Activation rates ===')
for col in ['rtrsvpor_active', 'rtrsvpoff_active', 'rtrdp_active']:
    rate = df[col].mean()
    print(f'  {col}: {rate:.3f} ({df[col].sum()} / {len(df)})')

print(f'\n=== Base energy ===')
print(f'  Negative: {(df["base_energy"] < 0).sum()} hours')
print(f'  < $1:     {(df["base_energy"] < 1).sum()} hours')
print(df['base_energy'].describe())

=== Activation rates ===
  rtrsvpor_active: 0.092 (3925 / 42858)
  rtrsvpoff_active: 0.043 (1847 / 42858)
  rtrdp_active: 0.158 (6782 / 42858)

=== Base energy ===
  Negative: 891 hours
  < $1:     1142 hours
count    42858.000000
mean        39.256552
std        120.977955
min       -307.804000
25%         17.634042
50%         25.021250
75%         39.642792
max       5014.400333
Name: base_energy, dtype: float64


### Sanity check: base_energy + total_adder ≈ rtm_price

In [5]:
residual = (df['base_energy'] + df['total_adder']) - df['rtm_price']
print(f'Max absolute residual: {residual.abs().max():.10f}')
assert residual.abs().max() < 1e-6, 'Decomposition check failed!'

Max absolute residual: 0.0000000000


### Export

In [6]:
export_cols = [
    'datetime', 'year', 'time_of_day', 'peak_hour', 'season',
    'spare_capacity', 'ng_price_monthly', 'degree_days', 'heating_dd', 'cap_threshold',
    'rtm_price', 'base_energy', 'total_adder',
    'RTRSVPOR', 'RTRSVPOFF', 'RTRDP',
    'rtrsvpor_active', 'rtrsvpoff_active', 'rtrdp_active',
]
df[export_cols].to_csv(OUT_PATH, index=False)
print(f'Exported {len(df)} rows to {OUT_PATH}')
print(f'Columns: {export_cols}')

Exported 42858 rows to 03_notebooks/03_model/../../01_data/4_model/v2_model_dataset.csv
Columns: ['datetime', 'year', 'time_of_day', 'peak_hour', 'season', 'spare_capacity', 'ng_price_monthly', 'degree_days', 'heating_dd', 'cap_threshold', 'rtm_price', 'base_energy', 'total_adder', 'RTRSVPOR', 'RTRSVPOFF', 'RTRDP', 'rtrsvpor_active', 'rtrsvpoff_active', 'rtrdp_active']


### Summary statistics for the Rmd

In [7]:
summary = pd.DataFrame({
    'col': df[export_cols].columns,
    'dtype': df[export_cols].dtypes.values,
    'missing': df[export_cols].isna().sum().values,
    'nunique': df[export_cols].nunique().values,
})
print(summary.to_string(index=False))
print()
print(df[export_cols].describe().to_string())

             col          dtype  missing  nunique
        datetime datetime64[us]        0    42849
            year          int64        0        5
     time_of_day          int64        0       24
       peak_hour           bool        0        2
          season            str        0        4
  spare_capacity        float64        0    42841
ng_price_monthly        float64        0       60
     degree_days        float64        0    41763
      heating_dd           bool        0        2
   cap_threshold           bool        0        2
       rtm_price        float64        0    40377
     base_energy        float64        0    40531
     total_adder        float64        0     3325
        RTRSVPOR        float64        0     1603
       RTRSVPOFF        float64        0      840
           RTRDP        float64        0     2507
 rtrsvpor_active          int64        0        2
rtrsvpoff_active          int64        0        2
    rtrdp_active          int64        0        2
